# 3. Taxonomy Classification
This notebook assigns taxonomic classifications using a custom-trained SILVA taxonomic classifier.

### Notebook Structure

**1.** Train a Taxonomic classifier    
&nbsp;&nbsp;&nbsp;&nbsp;**1.1** Preparing the SILVA reference database   
&nbsp;&nbsp;&nbsp;&nbsp;**1.2** Training an amplicon-region specific classifier   
**2.** Taxonomy assignment  
**3.** Evaluate taxonomic classifier 

### Import Packages

In [31]:
# Import all necessary packages
import os
import pandas as pd
from qiime2 import Visualization
import matplotlib.pyplot as plt
import numpy as np
import qiime2 as q2
%matplotlib inline

### Set Working Directory
Ensure that the working directory is correctly set to the 'scripts' folder within the main project directory. 
Otherwise, the file paths used in this notebook may not work properly.

In [32]:
# The working directory should normally default to the 'scripts' folder. 
# If it doesn't, set it manually using the command below.
# os.chdir("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts")  # Adjust this path to match your folder structure.

# Verify that your working directory is the 'scripts' folder inside the main project directory (.../MicrobiomeAnalysis_TummyTribe/scripts)
cwd = os.getcwd()
if not cwd.endswith("MicrobiomeAnalysis_TummyTribe/scripts"):
    print("WARNING: The working directory is not set to the 'scripts' folder inside 'MicrobiomeAnalysis_TummyTribe'!")
    print("Current working directory:", cwd)
    print("Please set it manually using os.chdir().")
else:
    print(f"Working directory is correctly set to the 'scripts' folder (\"{cwd}\").")

Working directory is correctly set to the 'scripts' folder ("/home/jovyan/Project/MicrobiomeAnalysis_TummyTribe/scripts").


In [33]:
# Data directories
raw_data_dir = "../data/raw"
meta_data_dir = "../data/processed/metadata"
denoised_data_dir = "../data/processed/denoising"
taxonomy_data_dir = "../data/processed/taxonomy"

In [9]:
%%bash -s "$taxonomy_data_dir"
mkdir -p "$1"

## 1. Train a Taxonomic classifier
We trained a custom Silva classifier on the V4 region of the 16S rRNA gene based on these two QIIME tutorials:
- [Processing, filtering, and evaluating the SILVA database (and other reference sequence data) with RESCRIPt](https://forum.qiime2.org/t/processing-filtering-and-evaluating-the-silva-database-and-other-reference-sequence-data-with-rescript/15494)
- [Using RESCRIPt's 'extract-seq-segments' to extract reference sequences without PCR primer pairs](https://forum.qiime2.org/t/using-rescripts-extract-seq-segments-to-extract-reference-sequences-without-pcr-primer-pairs/23618)
### 1.1 Preparing the SILVA reference database 
To reduce computation time and avoid memory issues on JupyterHub, the final step was run on Euler. The script can be found in the 'additional' folder (in scripts) under the name '3_euler_script_train_region_specific_classifier.sh'. The resulting classifier was uploaded to Polybox and then downloaded here, as the file was too large for GitHub.

Pipeline for Preparing the SILVA reference database and then training the classifier:
1. Download Silva reference (RNA) and reverse transcribe into DNA
2. Filter out poor quality 
3. Fiilter sequences by length and taxonomy
4. Dereplicate
5. Amplicon-region specific classifier
6. Train the classifier

In the next two cells we are downloading the **SILVA 138.2 SSURef NR99 RNA reference** database via QIIME RESCRIPt.

**SILVA 138.2** is a curated, high-quality database of ribosomal RNA sequences frequently used for 16S/18S taxonomic classification.

**SSU** → Small Subunit rRNA (in our case 16S rRNA)  
**Ref** → Refined   
**NR99** → Non-redundant at 99% identity (duplicate sequences are clustered so no two sequences are more than 99% identical).   

In [10]:
# Download Silva reference (RNA) --> missing permission on euler to download the data directly like this
! qiime rescript get-silva-data \
    --p-version '138.2' \
    --p-target 'SSURef_NR99' \
    --o-silva-sequences  $taxonomy_data_dir/silva-138.2-ssu-nr99-rna-seqs.qza \
    --o-silva-taxonomy $taxonomy_data_dir/silva-138.2-ssu-nr99-tax.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
^C
R[write to console]: 

Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/q2cli/util.py", line 275, in get_plugin_manager
    return qiime2.sdk.PluginManager.reuse_existing()
  File "/opt/conda/lib/python3.10/site-packages/qiime2/sdk/plugin_manager.py", line 58, in reuse_existing
    raise UninitializedPluginManagerError
qiime2.sdk.plugin_manager.UninitializedPluginManagerError

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/conda/bin/qiime", line 11, in <module>
    sys.exit(qiime())
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1161, in

In [12]:
# Reverse transcribe the RNA into DNA
! qiime rescript reverse-transcribe \
    --i-rna-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-rna-seqs.qza \
    --o-dna-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
^C

Aborted!


In the next two cells we filter out sequences with poor quality. Qiimes rescript cull-seqs filters sequences with more than 4 ambiguous bases (N’s, R, Y, etc.), sequences with homopolymers longer than 7 bp, sequences containing invalid characters and sequences with extremely short or degenerate content. As a next step we also filter the sequences by length to remove partial and abnormal sequences and taxonomy to remove invalid, missing, ambiguous, or irrelevant taxonomy.

In [13]:
# Filter out poor quality 
! qiime rescript cull-seqs \
    --i-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs.qza \
    --o-clean-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-cleaned.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
^C

Aborted!


In [19]:
# Filtering sequences by length and taxonomy
! qiime rescript filter-seqs-length-by-taxon \
    --i-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-cleaned.qza \
    --i-taxonomy $taxonomy_data_dir/silva-138.2-ssu-nr99-tax.qza \
    --p-labels Archaea Bacteria \
    --p-min-lens 900 1200 \
    --o-filtered-seqs $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-filt.qza \
    --o-discarded-seqs $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-discard.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[Sequence] to: ../data/processed/taxonomy/silva-138.2-ssu-nr99-seqs-filt.qza
Saved FeatureData[Sequence] to: ../data/processed/taxonomy/silva-138.2-ssu-nr99-seqs-discard.qza


In the next cells we dereplicate our sequences to ensure that each unique sequence only appears once. These unique reads are trimmed to only contain the V4 region, so that we can use it for our classifier. Since trimming can lead to duplicates again we dereplicate again.

In [20]:
# Dereplicate
! qiime rescript dereplicate \
    --i-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-filt.qza  \
    --i-taxa $taxonomy_data_dir/silva-138.2-ssu-nr99-tax.qza \
    --p-mode 'uniq' \
    --o-dereplicated-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-derep-uniq.qza \
    --o-dereplicated-taxa $taxonomy_data_dir/silva-138.2-ssu-nr99-tax-derep-uniq.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[Sequence] to: ../data/processed/taxonomy/silva-138.2-ssu-nr99-seqs-derep-uniq.qza
Saved FeatureData[Taxonomy] to: ../data/processed/taxonomy/silva-138.2-ssu-nr99-tax-derep-uniq.qza


In [21]:
# Make amplicon-region specific classifier
! qiime feature-classifier extract-reads \
    --i-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-derep-uniq.qza \
    --p-f-primer GTGYCAGCMGCCGCGGTAA \
    --p-r-primer GGACTACNVGGGTWTCTAAT \
    --p-n-jobs 2 \
    --p-read-orientation 'forward' \
    --o-reads $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-515f-806r.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[Sequence] to: ../data/processed/taxonomy/silva-138.2-ssu-nr99-seqs-515f-806r.qza


In [22]:
# Dereplicate again (could have new replicates in the shorter regions)
! qiime rescript dereplicate \
    --i-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-515f-806r.qza \
    --i-taxa $taxonomy_data_dir/silva-138.2-ssu-nr99-tax-derep-uniq.qza \
    --p-mode 'uniq' \
    --o-dereplicated-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-515f-806r-uniq.qza \
    --o-dereplicated-taxa $taxonomy_data_dir/silva-138.2-ssu-nr99-tax-515f-806r-derep-uniq.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[Sequence] to: ../data/processed/taxonomy/silva-138.2-ssu-nr99-seqs-515f-806r-uniq.qza
Saved FeatureData[Taxonomy] to: ../data/processed/taxonomy/silva-138.2-ssu-nr99-tax-515f-806r-derep-uniq.qza


### 1.2 Training an amplicon-region specific classifier

Now we need to train our amplicon-region specific classifier. There is not enough RAM available to run this in the notebook so we trained the classifier on euler. We added a wget to import the trained classifier. The euler scripts are available in the "additional" folder. The classifier is downloaded from Laura's Polybox into the taxonomy folder.

In [ ]:
# Train amplicon-region specific classifier 
"""
! qiime feature-classifier fit-classifier-naive-bayes \
    --i-reference-reads $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-515f-806r-uniq.qza \
    --i-reference-taxonomy $taxonomy_data_dir/silva-138.2-ssu-nr99-tax-515f-806r-derep-uniq.qza \
    --p-verbose \
    --o-classifier $taxonomy_data_dir/silva-138.2-ssu-nr99-515f-806r-classifier.qza
"""

In [5]:
%%bash -s "$taxonomy_data_dir"
mkdir -p "$1"

wget --progress=bar:force:noscroll \
  -O "$1/silva-138.2-ssu-nr99-515f-806r-classifier.qza" \
  "link to Polybox"

--2025-11-16 13:12:29--  https://polybox.ethz.ch/index.php/s/crQxtTa7MHXEKAk/download
Resolving polybox.ethz.ch (polybox.ethz.ch)... 129.132.71.243
Connecting to polybox.ethz.ch (polybox.ethz.ch)|129.132.71.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 63373448 (60M) [application/octet-stream]
Saving to: ‘../data/processed/taxonomy/silva-138.2-ssu-nr99-515f-806r-classifier.qza’

../data/processed/t 100%[===================>]  60.44M   185MB/s    in 0.3s    

2025-11-16 13:12:30 (185 MB/s) - ‘../data/processed/taxonomy/silva-138.2-ssu-nr99-515f-806r-classifier.qza’ saved [63373448/63373448]



## 2. Taxonomy assignment

As a next step we can use our trained classifier to assign taxonomy labels to our ASVs. And create some visualizations.

In [7]:
# Assign taxonomy labels to our ASVs 
! qiime feature-classifier classify-sklearn \
    --i-classifier $taxonomy_data_dir/silva-138.2-ssu-nr99-515f-806r-classifier.qza \
    --i-reads $denoised_data_dir/dada2_rep_seq.qza \
    --o-classification $taxonomy_data_dir/taxonomy.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[Taxonomy] to: ../data/processed/taxonomy/taxonomy.qza


In [8]:
# Check if it created the taxonomy artefact
! qiime tools peek $taxonomy_data_dir/taxonomy.qza

UUID:        ccecc755-3c0c-41af-8310-65e285f74e41
Type:        FeatureData[Taxonomy]
Data format: TSVTaxonomyDirectoryFormat


In [9]:
# Create the visualization
! qiime metadata tabulate \
    --m-input-file $taxonomy_data_dir/taxonomy.qza \
    --o-visualization $taxonomy_data_dir/taxonomy.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/taxonomy/taxonomy.qzv


In [10]:
Visualization.load(f"{taxonomy_data_dir}/taxonomy.qzv")

<visualization: Visualization uuid: 219c9cf6-4b8f-4551-b6c8-45bd87dbdcba>

In [11]:
# Create interactive taxonomy bar plot
! qiime taxa barplot \
    --i-table $denoised_data_dir/dada2_table.qza \
    --i-taxonomy $taxonomy_data_dir/taxonomy.qza \
    --m-metadata-file $meta_data_dir/metadata_merged.tsv \
    --o-visualization $taxonomy_data_dir/taxa-bar-plots.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/taxonomy/taxa-bar-plots.qzv


In [12]:
Visualization.load(f"{taxonomy_data_dir}/taxa-bar-plots.qzv")

<visualization: Visualization uuid: bfadcb56-e1b8-455b-8e99-42fcc4afba6f>

We can see that there are sequences from eukaryotes, mitochondria and chloroplasts in our data. So as a next step we will filter them out and have a look at the visualizations again.

In [27]:
! qiime taxa filter-table \
    --i-table {denoised_data_dir/dada2_table.qza \
    --i-taxonomy $taxonomy_data_dir/taxonomy.qza \
    --p-exclude Mitochondria,Chloroplast,Eukaryota \
    --o-filtered-table $taxonomydata_dir/dada2_table_filtered.qza


/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Usage: qiime taxa filter-table [OPTIONS]

  This method filters features from a table based on their taxonomic
  annotations. Features can be retained in the resulting table by specifying
  one or more include search terms, and can be filtered out of the resulting
  table by specifying one or more exclude search terms. If both include and
  exclude are provided, the inclusion critera will be applied before the
  exclusion critera. Either include or exclude terms (or both) must be
  provided. Any samples that have a total frequency of zero after filtering
  will be removed from the resulting table.

Inputs:
  --i-table ARTIFACT FeatureTable[Frequency¹ | PresenceAbs

In [28]:
cmd = f"""
qiime taxa filter-table \
  --i-table {denoised_data_dir}/dada2_table.qza \
  --i-taxonomy {taxonomy_data_dir}/taxonomy.qza \
  --p-exclude Mitochondria,Chloroplast,Eukaryota \
  --o-filtered-table {taxonomy_data_dir}/dada2_table_filtered.qza
"""

print(cmd)
!{cmd}


qiime taxa filter-table   --i-table ../data/processed/denoising/dada2_table.qza   --i-taxonomy ../data/processed/taxonomy/taxonomy.qza   --p-exclude Mitochondria,Chloroplast,Eukaryota   --o-filtered-table ../data/processed/taxonomy/dada2_table_filtered.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureTable[Frequency] to: ../data/processed/taxonomy/dada2_table_filtered.qza


In [26]:
! qiime taxa filter-seqs \
    --i-sequences $denoised_data_dir/dada2_rep_seq.qza \
    --i-taxonomy $taxonomy_data_dir/taxonomy.qza \
    --p-exclude Mitochondria,Chloroplast,Eukaryota \
    --o-filtered-sequences $taxonomy_data_dir/dada2_rep_seq_filtered.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[Sequence] to: ../data/processed/taxonomy/dada2_rep_seq_filtered.qza


In [34]:
# Create interactive taxonomy bar plot again
! qiime taxa barplot \
    --i-table $taxonomy_data_dir/dada2_table_filtered.qza \
    --i-taxonomy $taxonomy_data_dir/taxonomy.qza \
    --m-metadata-file $meta_data_dir/metadata_merged.tsv \
    --o-visualization $taxonomy_data_dir/taxa-bar-plots-filtered.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/taxonomy/taxa-bar-plots-filtered.qzv


In [35]:
Visualization.load(f"{taxonomy_data_dir}/taxa-bar-plots-filtered.qzv")

<visualization: Visualization uuid: 9fb55336-b321-4657-903a-f0519ca558f9>

In the barplots we don't see any eukaryotes so the filtering was successful.

We can quickly check how many sequences were identified up to which level.

In [36]:
# Load QIIME 2 artifact files as python objects
taxa = q2.Artifact.load(f'{taxonomy_data_dir}/taxonomy.qza')
# view as a `pandas.DataFrame`. Note: Only some Artifact types can be transformed to DataFrames
taxa = taxa.view(pd.DataFrame)

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [37]:
# Count for each taxonomic level how many ASVs were still identified
ranks = [("d__", "Domain"), ("p__", "Phylum"), ("c__", "Class"), ("o__", "Order"), ("f__", "Family"), ("g__", "Genus"), ("s__", "Species")]
total = len(taxa)
for i, (prefix, name) in enumerate(ranks, start=1):
    count = taxa["Taxon"].str.contains(prefix).sum()
    percent = count / total * 100
    print(f"{i}. {name}: {count} ({percent:.1f}%)")

1. Domain: 3873 (100.0%)
2. Phylum: 3870 (99.9%)
3. Class: 3868 (99.9%)
4. Order: 3855 (99.5%)
5. Family: 3790 (97.9%)
6. Genus: 3341 (86.3%)
7. Species: 3341 (86.3%)


Nearly all sequences were successfully classified, from 100% at the domain level to 86.3% at the genus and species levels. The identical genus and species counts are expected, since SILVA treats species identifications as extensions of the genus level, marked with the s__ prefix.

## 3. Evaluate taxonomic classifier

We trained two different classifiers and downloaded two pretrained ones from the internet to compare their performance. We then compared how many sequences each classifier could identify at different taxonomic levels.

The classifiers:

- **Region-specific:** A custom SILVA classifier trained on the V4 region of the 16S rRNA gene. (Code in this notebook & euler script for the last step in the 'additional' folder (in scripts) under the name '3_euler_script_train_region_specific_classifier.sh')

- **Full-length:** Identical to the region-specific classifier, but trained on the entire SILVA database rather than primer-trimmed sequences. This training was substantially more time-intensive, taking over 20 hours on Euler. (Euler script available in the scripts/additional folder under the name '3_euler_script_train_full_length_classifier.sh')

- **Pretrained classifier:** The classifier was downloaded from the [QIIME 2 data resources page](https://library.qiime2.org/data-resources). We chose a trained model weighted according to environment-specific (human stool) taxonomic abundance information, since this can significantly increase species-level classification accuracy across common sample types. [[Kaehler, 2019]](https://www.nature.com/articles/s41467-019-12669-6)
The classifier was also from SILVA.

    
We first need to download the pre-trained classifier and generate the taxonomic assignments for it.


In [14]:
# Dowload the pretrained, weighted – SILVA classifier
! wget -O $taxonomy_data_dir/silva-138-99-nb-human-stool-weighted-classifier.qza \
https://data.qiime2.org/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-human-stool-weighted-classifier.qza

--2025-11-16 13:14:55--  https://data.qiime2.org/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-human-stool-weighted-classifier.qza
Resolving data.qiime2.org (data.qiime2.org)... 54.200.1.12
Connecting to data.qiime2.org (data.qiime2.org)|54.200.1.12|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://s3-us-west-2.amazonaws.com/qiime2-data/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-human-stool-weighted-classifier.qza [following]
--2025-11-16 13:14:56--  https://s3-us-west-2.amazonaws.com/qiime2-data/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-human-stool-weighted-classifier.qza
Resolving s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)... 3.5.86.53, 52.218.253.48, 52.92.191.184, ...
Connecting to s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)|3.5.86.53|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 218311668 (208M) [binary/octet-stream]
Saving to: ‘../data/processed/taxonomy/silva-138-99-nb-human-st

In [15]:
# Predict the taxonomy with the pretrained silva classifier
! qiime feature-classifier classify-sklearn \
    --i-classifier $taxonomy_data_dir/silva-138-99-nb-human-stool-weighted-classifier.qza \
    --i-reads $denoised_data_dir/dada2_rep_seq.qza \
    --o-classification $taxonomy_data_dir/taxonomy_pretrained_silva.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[Taxonomy] to: ../data/processed/taxonomy/taxonomy_pretrained_silva.qza


Then we have to make sure we also have access to the full-length self-trained taxonomic classifier. We trained it on Euler and added it to the notebook via Polybox.

In [16]:
%%bash -s "$taxonomy_data_dir"
mkdir -p "$1"

# Training the full-length taxonomy classifier took more than 20 hours on Euler,
# so the script was added to the additional folder, and the classifier was uploaded to Polybox and downloaded here.
wget --progress=bar:force:noscroll \
  -O "$1/silva-138.2-ssu-nr99-classifier.qza" \
  "https://polybox.ethz.ch/index.php/s/oq4wjw2JKpqbwYb/download"

--2025-11-16 13:16:04--  https://polybox.ethz.ch/index.php/s/oq4wjw2JKpqbwYb/download
Resolving polybox.ethz.ch (polybox.ethz.ch)... 129.132.71.243
Connecting to polybox.ethz.ch (polybox.ethz.ch)|129.132.71.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 221286863 (211M) [application/octet-stream]
Saving to: ‘../data/processed/taxonomy/silva-138.2-ssu-nr99-classifier.qza’

../data/processed/t 100%[===================>] 211.04M   550MB/s    in 0.4s    

2025-11-16 13:16:04 (550 MB/s) - ‘../data/processed/taxonomy/silva-138.2-ssu-nr99-classifier.qza’ saved [221286863/221286863]



In [20]:
# Predict the taxonomy with the selftrained, full-length silva classifier
! qiime feature-classifier classify-sklearn \
    --i-classifier $taxonomy_data_dir/silva-138.2-ssu-nr99-classifier.qza \
    --i-reads $denoised_data_dir/dada2_rep_seq.qza \
    --o-classification $taxonomy_data_dir/taxonomy_custom_full_length.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[Taxonomy] to: ../data/processed/taxonomy/taxonomy_custom_full_length.qza


In [21]:
! qiime rescript evaluate-taxonomy \
  --i-taxonomies $taxonomy_data_dir/taxonomy.qza $taxonomy_data_dir/taxonomy_custom_full_length.qza $taxonomy_data_dir/taxonomy_pretrained_silva.qza\
  --p-labels region_specific full_length pretrained\
  --o-taxonomy-stats $taxonomy_data_dir/taxonomy-evaluation.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/taxonomy/taxonomy-evaluation.qzv


In [ ]:
Visualization.load(f"{taxonomy_data_dir}/taxonomy-evaluation.qzv")

# References

Kaehler, B.D., Bokulich, N.A., McDonald, D. et al. Species abundance information improves sequence taxonomy classification accuracy. Nat Commun 10, 4643 (2019). https://doi.org/10.1038/s41467-019-12669-6